# Data Import

In [28]:
import pandas as pd
import plotly.express as px
import branca.colormap as cm
import numpy as np
import folium
import numpy as np
from scipy import stats

In [29]:
accident2020_file = r"C:\Users\Clayg\OneDrive\Desktop\College\Fall 25 classes\Machine Learning\RawData\FARS2020NationalCSV\accident.csv"
accident2021_file = r"C:\Users\Clayg\OneDrive\Desktop\College\Fall 25 classes\Machine Learning\RawData\FARS2021NationalCSV\accident.csv"
accident2022_file = r"C:\Users\Clayg\OneDrive\Desktop\College\Fall 25 classes\Machine Learning\RawData\FARS2022NationalCSV\accident.csv"

In [30]:
accident2020 = pd.read_csv(accident2020_file, encoding='ISO-8859-1')
accident2021 = pd.read_csv(accident2021_file, encoding='ISO-8859-1')
accident2022 = pd.read_csv(accident2022_file, encoding='ISO-8859-1')

In [31]:
accident2020_cols = accident2020.columns.tolist()
accident2021_cols = accident2021.columns.tolist()
accident2022_cols = accident2022.columns.tolist()

In [32]:
print(f"All the columns in the dataframes are the same: {accident2020_cols == accident2021_cols == accident2022_cols}")

All the columns in the dataframes are the same: False


In [33]:
for col in accident2020_cols:
    if col not in accident2021_cols:
        print (f"Column {col} in 2020 is not in 2021")
    if col not in accident2022_cols:
        print (f"Column {col} in 2020 is not in 2022")

for col in accident2022_cols:
    if col not in accident2020_cols:
        print (f"Column {col} in 2021 is not in 2020")
    if col not in accident2022_cols:
        print (f"Column {col} in 2021 is not in 2022")

for col in accident2022_cols:
    if col not in accident2021_cols:
        print (f"Column {col} in 2022 is not in 2021")
    if col not in accident2020_cols:
        print (f"Column {col} in 2022 is not in 2020")

Column DRUNK_DR in 2020 is not in 2021
Column DRUNK_DR in 2020 is not in 2022


In [34]:
accident2020 = accident2020.drop("DRUNK_DR", axis=1)

In [35]:
combined_df = pd.concat([accident2020, accident2021, accident2022], axis=0, ignore_index=True)

In [36]:
combined_df.head()

,STATE,STATENAME,ST_CASE,VE_TOTAL,VE_FORMS,PVH_INVL,PEDS,PERNOTMVIT,PERMVIT,PERSONS,...,NOT_MINNAME,ARR_HOUR,ARR_HOURNAME,ARR_MIN,ARR_MINNAME,HOSP_HR,HOSP_HRNAME,HOSP_MN,HOSP_MNNAME,FATALS
0,1,Alabama,10001,1,1,0,0,0,4,4,...,Unknown,3,3:00am-3:59am,10,10,99,Unknown,99,Unknown EMS Hospital Arrival Time,3
1,1,Alabama,10002,4,4,0,0,0,6,6,...,18,17,5:00pm-5:59pm,26,26,99,Unknown,99,Unknown EMS Hospital Arrival Time,1
2,1,Alabama,10003,2,2,0,0,0,2,2,...,58,15,3:00pm-3:59pm,15,15,99,Unknown,99,Unknown EMS Hospital Arrival Time,1
3,1,Alabama,10004,1,1,0,0,0,5,5,...,Unknown,99,Unknown EMS Scene Arrival Hour,99,Unknown EMS Scene Arrival Minutes,99,Unknown,99,Unknown EMS Hospital Arrival Time,1
4,1,Alabama,10005,1,1,0,0,0,1,1,...,45,0,0:00am-0:59am,55,55,88,Not Applicable (Not Transported),88,Not Applicable (Not Transported),1


# Feature Engineering

In [37]:
state_mask = combined_df["STATENAME"] == "New York"
newyork_df = combined_df[state_mask]

In [38]:
len(newyork_df)

3161

In [39]:
newyork_df = newyork_df[["COUNTYNAME", "RUR_URB", "DAY_WEEK", "HOUR", "MINUTE", "FUNC_SYS", "LATITUDE", "LONGITUD", "WEATHER", "ARR_HOUR", "ARR_MIN", "HOSP_HR","HOSP_MN"]]

**Create function to calculate time differences**

First we'll drop all rows where an ambulance doesn't arrive or isn't known

In [40]:
ambulance_mask = newyork_df["ARR_HOUR"] <= 23
newyork_df = newyork_df[ambulance_mask]
hour_mask = newyork_df["HOUR"] <= 23
newyork_df = newyork_df[hour_mask]
long_lat_mask = newyork_df["LONGITUD"] != 999.9999
newyork_df = newyork_df[long_lat_mask]

In [41]:
newyork_df[["HOUR", "MINUTE"]]

,HOUR,MINUTE
21819,20,10
21821,20,26
21822,10,22
21823,6,41
21824,0,27
...,...,...
100572,18,11
100573,22,34
100574,21,26
100576,2,48


In [42]:
newyork_df['TIME_OBJ'] = pd.to_datetime(
    newyork_df['HOUR'].astype(str) + ":" + newyork_df['MINUTE'].astype(str), 
    format='%H:%M'
)

In [43]:
newyork_df['ARR_TIME_OBJ'] = pd.to_datetime(
    newyork_df['ARR_HOUR'].astype(str) + ":" + newyork_df['ARR_MIN'].astype(str), 
    format='%H:%M'
)

In [44]:
# 1. Perform the naive subtraction
# This will result in negative numbers for midnight crossovers (e.g., -23:50:00)
newyork_df['RESPONSE_TIME'] = newyork_df['ARR_TIME_OBJ'] - newyork_df['TIME_OBJ']

# 2. The "Midnight Correction"
# Find any row where the time difference is less than 0
negative_mask = newyork_df['RESPONSE_TIME'] < pd.Timedelta(0)

# Add exactly 1 day (24 hours) to ONLY those rows
newyork_df.loc[negative_mask, 'RESPONSE_TIME'] += pd.Timedelta(days=1)

# --- OPTIONAL: Convert to Minutes ---
# Right now the result is a "Timedelta" object (e.g., "0 days 00:10:00").
# If you want a simple number (10.0) for graphing:
newyork_df['RESPONSE_MINUTES'] = newyork_df['RESPONSE_TIME'].dt.total_seconds() / 60

# Check the specific midnight case
newyork_df[['TIME_OBJ', 'ARR_TIME_OBJ', 'RESPONSE_MINUTES']].sample(20)

,TIME_OBJ,ARR_TIME_OBJ,RESPONSE_MINUTES
99909,1900-01-01 22:47:00,1900-01-01 22:54:00,7.0
60852,1900-01-01 12:17:00,1900-01-01 12:27:00,10.0
60125,1900-01-01 19:52:00,1900-01-01 19:58:00,6.0
60752,1900-01-01 18:53:00,1900-01-01 19:09:00,16.0
100070,1900-01-01 20:08:00,1900-01-01 20:14:00,6.0
99868,1900-01-01 14:50:00,1900-01-01 14:54:00,4.0
22740,1900-01-01 18:05:00,1900-01-01 18:08:00,3.0
22321,1900-01-01 07:13:00,1900-01-01 07:19:00,6.0
22163,1900-01-01 13:36:00,1900-01-01 13:48:00,12.0
60073,1900-01-01 04:26:00,1900-01-01 04:27:00,1.0


Some of our response times seem too long, as many as 18 hours. To be safe we will remove this data on the assumption that it is faulty

In [45]:
long_wait_mask = newyork_df['RESPONSE_MINUTES'] < 240

newyork_df = newyork_df[long_wait_mask]

In [46]:
newyork_df.head()

,COUNTYNAME,RUR_URB,DAY_WEEK,HOUR,MINUTE,FUNC_SYS,LATITUDE,LONGITUD,WEATHER,ARR_HOUR,ARR_MIN,HOSP_HR,HOSP_MN,TIME_OBJ,ARR_TIME_OBJ,RESPONSE_TIME,RESPONSE_MINUTES
21819,OSWEGO (75),2,6,20,10,3,43.307281,-76.396225,2,20,15,99,99,1900-01-01 20:10:00,1900-01-01 20:15:00,0 days 00:05:00,5.0
21821,MONROE (55),2,6,20,26,1,43.044389,-77.555325,10,21,59,88,88,1900-01-01 20:26:00,1900-01-01 21:59:00,0 days 01:33:00,93.0
21822,SUFFOLK (103),2,6,10,22,2,40.744383,-73.280272,2,10,23,99,99,1900-01-01 10:22:00,1900-01-01 10:23:00,0 days 00:01:00,1.0
21823,MONROE (55),2,6,6,41,7,43.187144,-77.701492,1,6,47,88,88,1900-01-01 06:41:00,1900-01-01 06:47:00,0 days 00:06:00,6.0
21824,SARATOGA (91),2,6,0,27,1,42.817706,-73.767636,10,0,30,88,88,1900-01-01 00:27:00,1900-01-01 00:30:00,0 days 00:03:00,3.0


In [47]:
newyork_df["LATITUDE"] = newyork_df["LATITUDE"].astype(float)
newyork_df["LONGITUD"] = newyork_df["LONGITUD"].astype(float)

In [48]:
len_before = len(newyork_df)
newyork_df = newyork_df.dropna(subset=["LATITUDE", "LONGITUD"])

print(f"Dropped {len(newyork_df)-len_before} rows")

Dropped 0 rows


In [49]:
newyork_df[["LATITUDE", "LONGITUD"]].head(10)

,LATITUDE,LONGITUD
21819,43.307281,-76.396225
21821,43.044389,-77.555325
21822,40.744383,-73.280272
21823,43.187144,-77.701492
21824,42.817706,-73.767636
21826,40.915900,-72.675653
21828,43.222319,-77.600758
21830,44.983931,-74.565769
21831,40.847906,-73.366706
21832,44.477358,-73.596850


In [50]:
# 1. Filter out zero/negative values for log math (safety check)
# We use a small epsilon (1e-6) if there's a true 0, but usually response time > 0.
valid_times = newyork_df['RESPONSE_MINUTES'][newyork_df['RESPONSE_MINUTES'] > 0]
min_val = valid_times.min()
max_val = valid_times.max()

# 2. Create Logarithmic Indices
# This creates points that are spaced logarithmically (e.g., 1, 10, 100, 1000)
# Instead of Linear (0, 250, 500, 750, 1000)
log_index = np.geomspace(min_val, max_val, num=5)

# 3. Create the Colormap with these indices
# We use 5 colors to make the transition smoother across the log scale
colormap = cm.LinearColormap(
    colors=['green', 'lightgreen', 'yellow', 'orange', 'red'], 
    index=log_index,
    vmin=min_val, 
    vmax=max_val,
    caption='Response Time (Minutes) - Log Scale'
)

# 4. Create Base Map
map_center = [newyork_df["LATITUDE"].mean(), newyork_df["LONGITUD"].mean()]
m = folium.Map(location=map_center, zoom_start=7, control_scale=True)

# 5. Iterate and Plot
for _, row in newyork_df.iterrows():
    response_val = row['RESPONSE_MINUTES']
    
    if pd.isna(response_val) or response_val <= 0:
        c_color = "gray"
    else:
        c_color = colormap(response_val)

    folium.CircleMarker(
        location=[row["LATITUDE"], row["LONGITUD"]],
        radius=5,
        color=c_color,       
        fill=True,
        fill_color=c_color,  
        fill_opacity=0.7,
        popup=f"County: {row.get('COUNTYNAME', 'N/A')}<br>Mins: {response_val:.1f}"
    ).add_to(m)

# 6. Add Legend
m.add_child(colormap)

m

# Clustering by Latitude and Longitude to find distinct geographic regions

In [51]:
from sklearn.cluster import KMeans
import numpy as np

# Extract just the coordinates
X = newyork_df[["LATITUDE", "LONGITUD"]]

In [52]:
# 1. Initialize the model (n_clusters is the number of groups you want)
kmeans = KMeans(n_clusters=10, random_state=42)

# 2. Fit the model and predict the cluster for each point
# This adds a number (0, 1, 2, 3, 4) to each row
newyork_df["CLUSTER"] = kmeans.fit_predict(X)

# Check the results
print(newyork_df["CLUSTER"].value_counts())

CLUSTER
6    469
0    300
3    239
1    226
4    212
8    170
2    145
7     67
5     62
9     45
Name: count, dtype: int64


C:\Users\Clayg\miniconda3\envs\mlproj\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=8.
  warnings.warn(


In [53]:
cluster_colors = {
    0: 'red',
    1: 'blue',
    2: 'green',
    3: 'purple',
    4: 'orange',
    5: 'pink',
    6: 'lightblue',
    7: 'lightgreen',
    8:'black',
    9: 'white'
}

map_center = [newyork_df["LATITUDE"].mean(), newyork_df["LONGITUD"].mean()]
m = folium.Map(location=map_center, zoom_start=7) # Zoom out to see all clusters

for _, row in newyork_df.iterrows():
    
    # Get the cluster ID
    cluster_id = row["CLUSTER"]
    
    # Map ID to color (default to black if something goes wrong)
    c_color = cluster_colors.get(cluster_id, "black")
    
    folium.CircleMarker(
        location=[row["LATITUDE"], row["LONGITUD"]],
        radius=5,
        color=c_color,
        fill=True,
        fill_color=c_color,
        fill_opacity=0.7,
        popup=f"Cluster: {cluster_id}"
    ).add_to(m)

m

**Cluster Analysis**

In [54]:
newyork_df.head()

,COUNTYNAME,RUR_URB,DAY_WEEK,HOUR,MINUTE,FUNC_SYS,LATITUDE,LONGITUD,WEATHER,ARR_HOUR,ARR_MIN,HOSP_HR,HOSP_MN,TIME_OBJ,ARR_TIME_OBJ,RESPONSE_TIME,RESPONSE_MINUTES,CLUSTER
21819,OSWEGO (75),2,6,20,10,3,43.307281,-76.396225,2,20,15,99,99,1900-01-01 20:10:00,1900-01-01 20:15:00,0 days 00:05:00,5.0,4
21821,MONROE (55),2,6,20,26,1,43.044389,-77.555325,10,21,59,88,88,1900-01-01 20:26:00,1900-01-01 21:59:00,0 days 01:33:00,93.0,1
21822,SUFFOLK (103),2,6,10,22,2,40.744383,-73.280272,2,10,23,99,99,1900-01-01 10:22:00,1900-01-01 10:23:00,0 days 00:01:00,1.0,6
21823,MONROE (55),2,6,6,41,7,43.187144,-77.701492,1,6,47,88,88,1900-01-01 06:41:00,1900-01-01 06:47:00,0 days 00:06:00,6.0,1
21824,SARATOGA (91),2,6,0,27,1,42.817706,-73.767636,10,0,30,88,88,1900-01-01 00:27:00,1900-01-01 00:30:00,0 days 00:03:00,3.0,3


In [55]:
cluster_mean_response_times = newyork_df.groupby("CLUSTER")["RESPONSE_MINUTES"].mean()
cluster_std_response_times = newyork_df.groupby("CLUSTER")["RESPONSE_MINUTES"].std()
cluster_count_response_times = newyork_df['CLUSTER'].value_counts()
cluster_stats = pd.concat([cluster_mean_response_times, cluster_std_response_times, cluster_count_response_times], axis = 1)
cluster_stats

,RESPONSE_MINUTES,RESPONSE_MINUTES,count
CLUSTER,,,
0,13.256667,22.520243,300
1,9.730088,20.452062,226
2,13.027586,13.384249,145
3,11.389121,16.218675,239
4,11.783019,20.943392,212
5,15.064516,18.093447,62
6,9.569296,12.944738,469
7,19.208955,25.683704,67
8,10.205882,23.993070,170


In [56]:
X_cols = ['RUR_URB', 'DAY_WEEK', 'HOUR', 'FUNC_SYS', 'WEATHER']
y_cols = newyork_df['RESPONSE_MINUTES']

In [57]:
weather_map = {12:2, 5:3, 6:3, 7:3, 10:4}
hour_map = {0:1,2:1,3:1,4:1,5:1,6:2,7:2,8:2,9:2,10:2,11:2, 12:3, 13:3, 14:3, 15:3, 16:3, 17:3, 18:4, 19:4, 20:4, 21:4, 22:4, 23:4}

newyork_df['WEATHER'] = newyork_df['WEATHER'].replace(weather_map)
newyork_df["HOUR"] = newyork_df['HOUR'].replace(hour_map)

In [58]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [59]:
X_train, X_test, y_train, y_test = train_test_split(newyork_df[X_cols], y_cols, test_size=0.2, random_state=42)

In [67]:
preprocessor = ColumnTransformer(
    transformers=[
        # specific columns depending on your dataframe
        ('cat', OneHotEncoder(handle_unknown='ignore'), X_cols) 
    ],
    remainder='passthrough'
)

pipeline_RF = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

param_grid_RF = {
    'regressor__n_estimators': range(50, 151, 10),  # 50 to 150, step 10
    'regressor__min_samples_split': range(28, 42),    # 2 to 5
    'regressor__max_features': range(2, 12)         
}

grid_search_RF = GridSearchCV(
    estimator=pipeline_RF,
    param_grid=param_grid_RF,
    cv=5,                             
    scoring='neg_mean_squared_error', 
    n_jobs=-1,                        
    verbose=1
)

grid_search_RF.fit(X_train, y_train)
print(f"Best Parameters: {grid_search_RF.best_params_}")

Fitting 5 folds for each of 2200 candidates, totalling 11000 fits
Best Parameters: {'regressor__max_features': 3, 'regressor__min_samples_split': 30, 'regressor__n_estimators': 60}


In [68]:
best_rmse = np.sqrt(-1 * grid_search_RF.best_score_)
print(f"Best Cross-Validation RMSE: {best_rmse:.2f} minutes")

Best Cross-Validation RMSE: 17.87 minutes


In [64]:
y_mean = np.mean(y_test)
print(y_mean)

12.55813953488372
